## CS 180 AY 2026-2027 Mini-Project
### Names
### GERONIMO, Alexis

In [6]:
import csv
import os
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from pathlib import Path
import openpyxl

In [7]:
SEED: int = 12345678910
RAW_PSA = Path(".") / "data" / "raw" / "PSA"
RAW_FRED = Path(".") / "data" / "raw" / "FRED"
PROCESSED = Path(".") / "data" / "processed"

# print("current working directory:", os.getcwd())
# print("PSA dir:" , RAW_PSA.resolve())
# print("psa exists?", RAW_PSA.exists())

### CPI

In [8]:
df = pd.read_csv(RAW_PSA / "Year-on-Year Changes of the Core Consumer Price Index in Percent for All Items in the Philippines (2018=100)- January 1995 - December 2017 (Backcasted Values).csv", skiprows=1)
df = df[df["Period"] != "Average"].copy()
print(df.columns.tolist())
df_long = df.melt(
    id_vars=["Period"],
    var_name = "Year",
    value_name="ph_core_cpi"
)
df_long["date"] = pd.to_datetime(
    df_long["Year"] + " " + df_long["Period"],
    format = "%Y %B"
)
df_long = df_long[["date", "ph_core_cpi"]]
df_long = df_long.sort_values("date").reset_index(drop=True)
print(df_long.head(15))
df_long.to_csv(PROCESSED / "ph_core_cpi.csv", index=False)

['Period', '1995', '1996', '1997', '1998', '1999', '2000', '2001', '2002', '2003', '2004', '2005', '2006', '2007', '2008', '2009', '2010', '2011', '2012', '2013', '2014', '2015', '2016', '2017']
         date  ph_core_cpi
0  1995-01-01          6.3
1  1995-02-01          5.9
2  1995-03-01          6.3
3  1995-04-01          5.8
4  1995-05-01          6.4
5  1995-06-01          6.4
6  1995-07-01          6.2
7  1995-08-01          6.1
8  1995-09-01          6.1
9  1995-10-01          6.7
10 1995-11-01          6.7
11 1995-12-01          6.5
12 1996-01-01          6.7
13 1996-02-01          6.7
14 1996-03-01          6.8


### RPI

In [16]:
df = pd.read_excel(RAW_PSA / "Income and Outlay Account in Households including NPISHs_1.xlsx", sheet_name="HH IO", header=None)
years = df.iloc[9, 1:27].astype(int).values
income = pd.to_numeric(
    df.iloc[23, 1:27],
    errors="coerce"
).values

annual_income = pd.DataFrame({
    "year": years,
    "ph_total_household_income": income
})
print(annual_income.head())

monthly_income = np.repeat(annual_income["ph_total_household_income"].values, 12)
dates = pd.date_range(
    start="2000-01-01",
    periods=len(monthly_income),
    freq="MS"
)

ph_rpi = pd.DataFrame({
    "date": dates,
    "ph_total_household_income": monthly_income
})

ph_rpi.to_csv(PROCESSED / "ph_total_household_income_monthly.csv", index=False)
print(ph_rpi.head(15))


   year  ph_total_household_income
0  2000               3.358298e+06
1  2001               3.631655e+06
2  2002               3.968051e+06
3  2003               4.472276e+06
4  2004               4.860831e+06
         date  ph_total_household_income
0  2000-01-01               3.358298e+06
1  2000-02-01               3.358298e+06
2  2000-03-01               3.358298e+06
3  2000-04-01               3.358298e+06
4  2000-05-01               3.358298e+06
5  2000-06-01               3.358298e+06
6  2000-07-01               3.358298e+06
7  2000-08-01               3.358298e+06
8  2000-09-01               3.358298e+06
9  2000-10-01               3.358298e+06
10 2000-11-01               3.358298e+06
11 2000-12-01               3.358298e+06
12 2001-01-01               3.631655e+06
13 2001-02-01               3.631655e+06
14 2001-03-01               3.631655e+06


### Labor Force, Employed, and Unemployed

In [38]:
df = pd.read_csv(RAW_PSA / "Levels of Key Employment Indicators- April 2005 to September 2026.csv", skiprows=1, encoding="utf-8-sig")
labor = df.iloc[:, [0, 1, 5, 8, 11]].copy()

labor.columns = [
    "year",
    "month",
    "ph_labor_force",
    "ph_employed",
    "ph_unemployed"
]
labor["month"] = labor["month"].astype(str).str.strip()
labor = labor[labor["month"] != "Annual"].copy()

value_columns = [
    "ph_labor_force",
    "ph_employed",
    "ph_unemployed"
]

labor[value_columns] = labor[value_columns].replace(".", pd.NA)
labor[value_columns] = labor[value_columns].apply(
    pd.to_numeric, errors="coerce"
)

labor["date"] = pd.to_datetime(
    labor["year"].astype(str) + " " + labor["month"],
    format="%Y %B",
    errors="coerce"
)

labor = labor.dropna(subset=["date"] + value_columns)

labor = labor[
    ["date", "ph_labor_force", "ph_employed", "ph_unemployed"]
]

labor = labor.sort_values("date").reset_index(drop=True)
labor.to_csv(PROCESSED / "ph_labor_market.csv", index=False)
print(labor.head())

        date  ph_labor_force  ph_employed  ph_unemployed
0 2005-04-01       35129.827    32221.085       2908.742
1 2005-07-01       35236.487    32521.695       2714.792
2 2005-10-01       35494.050    32875.281       2618.769
3 2006-01-01       34868.478    32031.302       2837.176
4 2006-04-01       35633.475    32698.970       2934.505


### Consolidating all datasets

In [ ]:
cpi = pd.read_csv(PROCESSED / "ph_core_cpi.csv")
income = pd.read_csv(PROCESSED / "ph_total_household_income_monthly.csv")
labor = pd.read_csv(PROCESSED / "ph_labor_market.csv")
cpi = cpi[["date", "ph_core_cpi"]].copy()
income= income[["date", "ph_total_household_income"]].copy()
labor=labor[["date", "ph_labor_force", "ph_employed", "ph_unemployed", "ph_unemployed_rate"]].copy()
for dataset in [cpi, income, labor]:
    dataset["date"] = pd.to_datetime(dataset["date"])

master = cpi.merge(
    income,
    on="date",
    how="outer",
    validate="one_to_one"
)

master = master.merge(
    labor,
    on="date",
    how="outer",
    validate="one_to_one"
)

master = master.sort_values("date").reset_index(drop=True)

master.to_csv(
    PROCESSED / "philippines_master.csv",
    index=False
)

print(master.head(10))
print(master.tail(10))
print(master.info())

KeyError: "['ph_unemployed_rate'] not in index"